# Campanha de pesquisa no Colab

Roda várias fatias (`--shard I/N`) da grade do `run_research.py` **ao mesmo tempo**, com um
único Ollama na GPU atendendo as requisições em paralelo. Cada fatia contém blocos pareados
completos e grava em uma pasta própria no Google Drive. No fim, `analysis/analyze_completion.py`
recebe as pastas de todas as fatias, inclusive as da máquina local.

Regras para a consolidação ser aceita:

- Todas as máquinas usam o **mesmo commit**, o mesmo modelo (digest conferido abaixo),
  o mesmo `SEED_COUNT` e o mesmo `N_SHARDS`.
- Cada fatia roda em **uma só máquina**. A local fica com as fatias que não estiverem em `SHARDS`.
- Execute as fatias em ordem crescente; parar por tempo é aceitável, escolher fatias pelo resultado não.
- Ambiente com GPU (Runtime > Change runtime type). L4 ou A100 comportam mais fatias em paralelo que a T4.
- Repositório privado: crie em Secrets (ícone de chave) o segredo `GITHUB_TOKEN`, com um token de leitura do repositório.
  A célula do clone precisa ser executada pela interface do Colab: segredos não são lidos por execução remota.
- Sem execução em segundo plano no plano Pro, mantenha a aba aberta.

In [ ]:
# Ajuste COMMIT e SHARDS; SEED_COUNT e N_SHARDS devem ser iguais aos da máquina local.
COMMIT = ""  # saída de `git rev-parse HEAD` na máquina local
SEED_COUNT = 24  # 4.224 casos; igual ao padrão do run_research.py
N_SHARDS = 8  # três sementes e 528 casos por fatia; a máquina local fica com a fatia 0
SHARDS = [1, 2, 3]  # fatias rodadas aqui, em paralelo
HOURS = 20  # orçamento desta sessão; renova a cada retomada
BASE_DIR = "/content/drive/MyDrive/brazilianinfosim/conclusao"

MODEL = "qwen3:4b-instruct-2507-q4_K_M"
EXPECTED_DIGEST = "0edcdef34593eac1aa2be9c7d06c432dcf81945adca5eca2f27662c18f168ba0"
OLLAMA_VERSION = "0.20.5"
PYTHON_VERSION = "3.14.3"
REPO = "yrlanrteixeira/brazilianinfosim"
assert COMMIT, "Preencha COMMIT com o mesmo commit da máquina local."
assert SEED_COUNT % N_SHARDS == 0 and all(0 <= s < N_SHARDS for s in SHARDS)

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import base64, subprocess, urllib.error, urllib.request
token = userdata.get("GITHUB_TOKEN").strip()
# Token no cabeçalho, fora da URL, para não aparecer em mensagens de erro.
request = urllib.request.Request(f"https://api.github.com/repos/{REPO}", headers={"Authorization": f"Bearer {token}"})
try:
    urllib.request.urlopen(request)
except urllib.error.HTTPError as error:
    raise RuntimeError({401: "Token inválido ou expirado.",
                        404: "Token sem acesso ao repositório: confira o dono e o repositório selecionado."}.get(error.code, error.code))
header = "Authorization: Basic " + base64.b64encode(f"x-access-token:{token}".encode()).decode()
subprocess.run(["rm", "-rf", "/content/brazilianinfosim"], check=True)
for command in (["git", "-c", f"http.extraHeader={header}", "clone", "-q", f"https://github.com/{REPO}.git", "/content/brazilianinfosim"],
                ["git", "-C", "/content/brazilianinfosim", "checkout", "-q", COMMIT]):
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(result.stderr.replace(token, "***"))
!git -C /content/brazilianinfosim log --oneline -1


O ambiente Python fica isolado do Colab, com a mesma versão do Python e os pacotes fixados
em `requirements.txt`. O PyTorch de CPU basta: a campanha não calcula embeddings.

In [ ]:
%cd /content/brazilianinfosim
!pip install -q uv
!uv python install {PYTHON_VERSION}
!uv venv -q --clear --python {PYTHON_VERSION} /content/venv
!uv pip install -q --python /content/venv/bin/python -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cpu --index-strategy unsafe-best-match
!/content/venv/bin/python --version

In [ ]:
import json, os, time, urllib.request
!curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION={OLLAMA_VERSION} sh > /dev/null
# Uma vaga de inferência por fatia; o modelo fica carregado durante toda a sessão.
env = {**os.environ, "OLLAMA_NUM_PARALLEL": str(len(SHARDS)), "OLLAMA_MAX_LOADED_MODELS": "1", "OLLAMA_KEEP_ALIVE": "-1"}
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
!ollama pull {MODEL}
tags = json.load(urllib.request.urlopen("http://localhost:11434/api/tags"))["models"]
version = json.load(urllib.request.urlopen("http://localhost:11434/api/version"))["version"]
digest = next(m["digest"] for m in tags if m["name"] == MODEL)
print(version, digest)
assert digest == EXPECTED_DIGEST, "Modelo diferente do local; as fatias não poderão ser juntadas."
assert version == OLLAMA_VERSION, "Versão do Ollama diferente da combinada."

Inicia uma execução por fatia e acompanha o progresso. Se a sessão cair, rode de novo todas
as células: casos completos não se repetem e falhas ou interrupções são retentadas.

In [ ]:
from pathlib import Path
os.makedirs(f"{BASE_DIR}/logs", exist_ok=True)
# O Colab exporta um backend inline do matplotlib que o Python isolado não aceita.
worker_env = {**os.environ, "MPLBACKEND": "Agg"}
workers = {}
for shard in SHARDS:
    folder = Path(BASE_DIR, f"shard-{shard}-de-{N_SHARDS}")
    command = ["/content/venv/bin/python", "scripts/run_research.py", "--output", str(folder),
               "--seed-count", str(SEED_COUNT), "--shard", f"{shard}/{N_SHARDS}", "--hours", str(HOURS)]
    if (folder / "campaign.json").exists():
        command += ["--resume", "--retry-incomplete"]
    log = open(f"{BASE_DIR}/logs/shard-{shard}-de-{N_SHARDS}.log", "a")
    workers[shard] = (folder, subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT,
                                               cwd="/content/brazilianinfosim", env=worker_env))


In [ ]:
def report():
    for shard, (folder, process) in workers.items():
        status_path = folder / "status.json"
        status = json.loads(status_path.read_text(encoding="utf-8")) if status_path.exists() else {}
        state = "rodando" if process.poll() is None else f"saiu com {process.returncode}"
        print(f"fatia {shard}: {status.get('success', 0)}/{status.get('planned', '?')} sucesso, "
              f"{status.get('failed', 0)} falhas, {status.get('complete_blocks', 0)} blocos, {state}", flush=True)

report()


In [ ]:
# Painel em tempo real num processo separado: não ocupa o kernel e só abre no seu navegador.
PAINEL = r'''
import json, re, subprocess, sys, time
from datetime import datetime
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
from statistics import mean

BASE, N = Path(sys.argv[1]), int(sys.argv[2])
SHARDS = [int(s) for s in sys.argv[3].split(",")]
GIN = re.compile(r"\[GIN\] (\d{4}/\d{2}/\d{2} - \d{2}:\d{2}:\d{2}) \| 200 .*/v1/chat/completions")
cache = {}

def read(path):
    stamp = path.stat().st_mtime
    if cache.get(path, (None,))[0] != stamp:
        cache[path] = (stamp, json.loads(path.read_text(encoding="utf-8")))
    return cache[path][1]

def collect():
    now = time.time()
    with open("/content/ollama.log", encoding="utf-8", errors="ignore") as log:
        stamps = [datetime.strptime(m[1], "%Y/%m/%d - %H:%M:%S").timestamp() for m in map(GIN.search, log) if m]
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
                          "--format=csv,noheader,nounits"], capture_output=True, text=True).stdout.split(",")
    shards, records = [], []
    for shard in SHARDS:
        folder = BASE / f"shard-{shard}-de-{N}"
        status = read(folder / "status.json") if (folder / "status.json").exists() else {}
        sessions = list((folder / "budget_sessions").glob("*.json")) if (folder / "budget_sessions").exists() else []
        deadline = max((read(p)["deadline_epoch"] for p in sessions), default=None)
        for path in (folder / "attempts").glob("*.json") if (folder / "attempts").exists() else []:
            r, summary = read(path), read(path).get("summary") or {}
            records.append({"shard": shard, "family": r["family"], "condition": r["condition"], "status": r["status"],
                            "reach": summary.get("final_reach_pct"), "ticks": summary.get("ticks_run"),
                            "stop": summary.get("stop_reason"), "calls": (r.get("llm_stats") or {}).get("api_calls"),
                            "seconds": r.get("elapsed_seconds"), "finished": path.stat().st_mtime})
        shards.append({"shard": shard, "planned": status.get("planned", 0), "success": status.get("success", 0),
                       "failed": status.get("failed", 0), "interrupted": status.get("interrupted", 0),
                       "blocks": status.get("complete_blocks", 0), "status": status.get("status", "iniciando"),
                       "hours_left": (deadline - now) / 3600 if deadline else None})
    done = [r for r in records if r["status"] == "success"]
    per_minute = sum(s > now - 600 for s in stamps) / 10
    mean_calls = mean(r["calls"] for r in done if r["calls"] is not None) if done else None
    remaining = sum(s["planned"] for s in shards) - len(done)
    return {"updated": now, "shards": shards,
            "recent": sorted(records, key=lambda r: -r["finished"])[:12],
            "llm": {"total": len(stamps), "last_minute": sum(s > now - 60 for s in stamps), "per_minute_10": per_minute},
            "gpu": {"util": float(gpu[0]), "used": float(gpu[1]), "total": float(gpu[2])} if len(gpu) == 3 else None,
            "summary": {"done": len(done), "planned": sum(s["planned"] for s in shards), "mean_calls": mean_calls,
                        "mean_minutes": mean(r["seconds"] for r in done) / 60 if done else None,
                        "eta_hours": remaining * mean_calls / per_minute / 60 if mean_calls and per_minute else None}}

PAGE = """<!doctype html><html lang="pt-BR"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1"><title>Campanha no Colab</title><style>
:root{--bg:#f6f6f4;--card:#fff;--fg:#1c1c1e;--muted:#6b6b72;--line:#e2e2de;--accent:#2563eb;--ok:#15803d;--bad:#b91c1c;--warn:#b45309}
@media (prefers-color-scheme:dark){:root{--bg:#131315;--card:#1c1c1f;--fg:#ececef;--muted:#9a9aa3;--line:#2d2d33;--accent:#60a5fa;--ok:#4ade80;--bad:#f87171;--warn:#fbbf24}}
body{margin:0;background:var(--bg);color:var(--fg);font:14px/1.45 system-ui,-apple-system,"Segoe UI",sans-serif}
main{max-width:1100px;margin:0 auto;padding:20px 16px}
h1{font-size:18px;margin:0}header{display:flex;justify-content:space-between;align-items:baseline;gap:12px;flex-wrap:wrap;margin-bottom:16px}
.muted{color:var(--muted)}.kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:10px;margin-bottom:14px}
.card{background:var(--card);border:1px solid var(--line);border-radius:10px;padding:12px 14px}
.kpi b{display:block;font-size:22px;font-variant-numeric:tabular-nums}.kpi span{color:var(--muted);font-size:12px}
.shards{display:grid;grid-template-columns:repeat(auto-fit,minmax(240px,1fr));gap:10px;margin-bottom:14px}
.bar{height:8px;background:var(--line);border-radius:4px;overflow:hidden;margin:8px 0}.bar i{display:block;height:100%;background:var(--accent)}
.row{display:flex;justify-content:space-between;font-size:12px;color:var(--muted)}
.wrap{overflow-x:auto}table{width:100%;border-collapse:collapse;font-variant-numeric:tabular-nums}
th,td{text-align:left;padding:6px 8px;border-bottom:1px solid var(--line);white-space:nowrap}th{color:var(--muted);font-weight:500;font-size:12px}
.success{color:var(--ok)}.failed{color:var(--bad)}.interrupted{color:var(--warn)}
</style></head><body><main>
<header><h1>Campanha de pesquisa no Colab</h1><span class="muted" id="updated">carregando…</span></header>
<section class="kpis" id="kpis"></section><section class="shards" id="shards"></section>
<section class="card"><div class="row" style="margin-bottom:6px"><b style="color:var(--fg)">Casos concluídos recentemente</b></div>
<div class="wrap"><table><thead><tr><th>Fatia</th><th>Família</th><th>Condição</th><th>Status</th><th>Alcance</th><th>Rodadas</th><th>Chamadas</th><th>Duração</th><th>Fim</th></tr></thead>
<tbody id="recent"></tbody></table></div></section></main><script>
const f=(v,d=0)=>v==null?"–":Number(v).toFixed(d);
const clock=t=>new Date(t*1000).toLocaleTimeString("pt-BR");
async function tick(){try{const d=await (await fetch("dados")).json(),s=d.summary;
document.getElementById("updated").textContent="atualizado às "+clock(d.updated);
const k=[["Casos concluídos",s.done+" / "+s.planned],["Chamadas no último minuto",d.llm.last_minute],
["Chamadas por minuto (10 min)",f(d.llm.per_minute_10,1)],["Chamadas por caso",f(s.mean_calls)],
["Minutos por caso",f(s.mean_minutes,1)],["GPU",d.gpu?f(d.gpu.util)+"% · "+f(d.gpu.used/1024,1)+" GB":"–"],
["Estimativa restante",s.eta_hours==null?"aguardando casos":f(s.eta_hours,1)+" h"]];
document.getElementById("kpis").innerHTML=k.map(([a,b])=>`<div class="card kpi"><b>${b}</b><span>${a}</span></div>`).join("");
document.getElementById("shards").innerHTML=d.shards.map(x=>`<div class="card"><b>Fatia ${x.shard}</b> <span class="muted">${x.status}</span>
<div class="bar"><i style="width:${x.planned?100*x.success/x.planned:0}%"></i></div>
<div class="row"><span>${x.success}/${x.planned} casos · ${x.blocks}/${x.planned/22} blocos</span><span>${x.failed} falhas · ${x.interrupted} interr.</span></div>
<div class="row"><span>orçamento restante</span><span>${x.hours_left==null?"–":f(x.hours_left,1)+" h"}</span></div></div>`).join("");
document.getElementById("recent").innerHTML=d.recent.map(r=>`<tr><td>${r.shard}</td><td>${r.family}</td><td>${r.condition}</td>
<td class="${r.status}">${r.status}</td><td>${f(r.reach)}%</td><td>${f(r.ticks)}</td><td>${f(r.calls)}</td><td>${f(r.seconds/60,1)} min</td><td>${clock(r.finished)}</td></tr>`).join("")
||'<tr><td colspan="9" class="muted">Nenhum caso concluído ainda.</td></tr>';
}catch(e){document.getElementById("updated").textContent="sem resposta: "+e}}
tick();setInterval(tick,5000);
</script></body></html>"""

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        body, kind = (json.dumps(collect()).encode(), "application/json") if self.path.endswith("dados") else (PAGE.encode(), "text/html; charset=utf-8")
        self.send_response(200); self.send_header("Content-Type", kind); self.end_headers(); self.wfile.write(body)
    def log_message(self, *args):
        pass

ThreadingHTTPServer(("0.0.0.0", int(sys.argv[4])), Handler).serve_forever()
'''
Path("/content/painel.py").write_text(PAINEL, encoding="utf-8")
PORT = 8765
subprocess.run(["pkill", "-f", "/content/painel.py"])
painel = subprocess.Popen(["python3", "/content/painel.py", BASE_DIR, str(N_SHARDS), ",".join(map(str, SHARDS)), str(PORT)],
                          stdout=open("/content/painel.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(2)
from google.colab import output
output.serve_kernel_port_as_window(PORT, anchor_text="Abrir painel da campanha")


In [ ]:
# Acompanhamento contínuo; interromper esta célula não para as fatias.
while any(process.poll() is None for _, process in workers.values()):
    time.sleep(600)
    print(time.strftime("%H:%M"), flush=True)
    report()


In [ ]:
# Diagnóstico de uma fatia que saiu com erro.
!tail -n 30 "{BASE_DIR}/logs/shard-{SHARDS[0]}-de-{N_SHARDS}.log"